<a href="https://colab.research.google.com/github/liqq1024/GenAI_LLMs_2026Fall/blob/main/colab-notebooks/Week5_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Prompt Engineering and LLM Interaction

## Live Coding Demonstration (Google Colab)

Prompt engineering is systematic natural-language programming: specifying a task, context, constraints, and desired output well enough for a language model to produce useful results.

### Objectives

By the end of this activity, you will be able to:

- use zero-shot, few-shot, role, structured-output, and reasoning-oriented prompting;
- compare responses from different available models;
- generate and validate structured JSON output;
- evaluate prompt quality using explicit criteria; and
- iteratively refine a weak prompt.

**Estimated time:** 35–45 minutes

> Do not paste an API key into a notebook cell or save it in a shared copy. Use the hidden input cell below.


## Section 1: Load a Chat Model

In [7]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "openai", "pandas", "huggingface_hub"])

import json
import os
from getpass import getpass

import pandas as pd
from openai import OpenAI
from huggingface_hub import InferenceClient

#for OpenAI
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("Enter your API key (input is hidden): ")
#for Huggingface
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Enter your Huggingface token (input is hidden): ")


# #for OpenAI
# client = OpenAI()

#for Huggingface
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"]
)

# Use a model available to your account. Add a second available model to compare outputs.
MODEL_NAMES = ["zai-org/GLM-5.3-Flash"]

#for  OpenAI
# def ask_model(prompt, model_name, instructions=None):
#     response = client.responses.create(
#         model=model_name,
#         instructions=instructions,
#         input=prompt,
#     )
#     return response.output_text

#for Huggingface
def ask_model(prompt, model_name, instructions=None):
    messages = []
    if instructions:
        messages.append({"role": "system", "content": instructions})
    messages.append({"role": "user", "content": prompt})

    response = client.chat_completion(
        model=model_name,
        messages=messages,
        max_tokens=400,
        temperature=0.2,
    )
    return response.choices[0].message.content

print("Configured models:", MODEL_NAMES)

Configured models: ['zai-org/GLM-5.3-Flash']


In [9]:
import os

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("❌ HF_TOKEN is NOT set in the environment variables.")
else:
    print("✅ HF_TOKEN is set in the environment.")
    print(f"Token length: {len(hf_token)}")
    if hf_token.startswith("hf_"):
        print("✅ Token starts with 'hf_' (standard Hugging Face token).")
    else:
        print("⚠️ Warning: Token does not start with 'hf_'. It starts with:", hf_token[:5] + "...")

✅ HF_TOKEN is set in the environment.
Token length: 37
✅ Token starts with 'hf_' (standard Hugging Face token).


### A simple comparison helper

Run this cell after each prompting activity to compare the same prompt across all available models. If only one model is configured, use the later comparison questions to compare different prompting strategies instead.

In [10]:
def compare_models(prompt, instructions=None):
    results = []
    for model_name in MODEL_NAMES:
        results.append(
            {
                "model": model_name,
                "response": ask_model(prompt, model_name, instructions),
            }
        )
    return pd.DataFrame(results)


def show_comparison(prompt, instructions=None):
    table = compare_models(prompt, instructions)
    for _, row in table.iterrows():
        print("\n" + "=" * 80)
        print("MODEL:", row["model"])
        print(row["response"])
    return table

## Section 2: Zero-Shot Prompting

A zero-shot prompt gives the model an instruction without worked examples.

Run the prompt, then improve it by adding a clear audience, constraints, and output format. Compare the two responses.

In [11]:
zero_shot_prompt = """
Explain why causal masking is important in a decoder-only language model.
"""

show_comparison(zero_shot_prompt)

# TODO: Rewrite the prompt for a senior undergraduate CS student.
# Require a short answer, one concrete example, and no more than 120 words.
# improved_zero_shot_prompt = """
# Explain why causal masking is important in a decoder-only language model for a senior UG CS student. Give one concrete example with concise explanation no more than 120 words
# """

# show_comparison(improved_zero_shot_prompt)


MODEL: zai-org/GLM-5.3-Flash
# Causal Masking in Decoder-Only Language Models

## The Core Idea

Causal masking (also called an attention mask or look-ahead mask) prevents each token from attending to any tokens that come **after** it in the sequence. When token *i* computes its attention, it can only see tokens at positions 1 through *i* — never *i+1* onward.

## Why It's Necessary

### 1. The training objective requires it
Decoder-only models (like GPT) are trained with **next-token prediction**: given tokens 1 through *i*, predict token *i+1*. If a token could attend to future positions during training, the model could simply "copy" the answer from the future — the task becomes trivially easy, gradients become useless, and the model learns nothing meaningful. The mask preserves the actual prediction problem.

### 2. Consistency between training and inference
At inference time, the model generates tokens **one at a time, left to right**. When generating token *i+1*, the future token

,model,response
0,zai-org/GLM-5.3-Flash,# Causal Masking in Decoder-Only Language Mode...


**Discuss:** Which details of your revised prompt most improved the response? Did more detail make the answer more accurate, more focused, or merely longer?

## Section 3: Few-Shot Prompting

Few-shot prompting provides examples of the input-output pattern before a new input. Here, the model classifies whether a prompt is sufficiently specific for a student-facing assignment.

In [ ]:
few_shot_prompt = """
Classify each prompt as Strong or Weak. Then give one short reason.

Prompt: Explain attention.
Classification: Weak
Reason: It does not specify the audience, depth, or output form.

Prompt: For a senior undergraduate CS student, explain self-attention in 100 words. Include Q, K, and V, and use one sentence-level example.
Classification: Strong
Reason: It specifies audience, scope, constraints, and an example requirement.

Prompt: Tell me about transformer models.
Classification:
Reason:
"""

few_shot_results = show_comparison(few_shot_prompt)

# TODO: Add one more Strong/Weak demonstration example and classify a new prompt.

**Discuss:** What pattern did the examples teach the model? When could examples accidentally bias the model toward an undesirable answer pattern?

## Section 4: Role Prompting

A role instruction sets an expected perspective, expertise level, or communication style. It should support the task; it does not replace a clear task description.

In [ ]:
role_instructions = """
You are a patient teaching assistant for an introductory-to-intermediate
computer science course. Use precise but accessible language. Do not invent facts.
"""

role_prompt = """
Explain the difference between a token ID and an embedding.
Use two short paragraphs and one analogy.
"""

role_results = show_comparison(role_prompt, instructions=role_instructions)

# TODO: Run the same task without role_instructions and compare the answers.
# show_comparison(role_prompt)

**Discuss:** Which aspects of the response changed because of the role instruction? Did the role improve correctness, presentation, or both?

## Section 5: Structured JSON Output

Applications often need outputs that code can read reliably. Ask for JSON only, then validate it with Python. If parsing fails, refine the prompt rather than manually repairing the model output.

In [ ]:
json_prompt = """
Return JSON only. Do not use Markdown fences or explanatory text.

Create a study plan for learning causal masking. The JSON object must have exactly:
- topic: a string
- learning_goals: a list of exactly 3 strings
- practice_activity: a string
- estimated_minutes: an integer between 15 and 60
"""

json_response = ask_model(json_prompt, MODEL_NAMES[0])
print(json_response)

try:
    study_plan = json.loads(json_response)
    print("\nValid JSON:")
    print(json.dumps(study_plan, indent=2))
except json.JSONDecodeError as error:
    print("\nThe response was not valid JSON:", error)
    print("TODO: Refine the prompt so that it returns JSON only.")

**Discuss:** Why is valid JSON useful in an application pipeline? What additional validation should code perform even after JSON parsing succeeds?

## Section 6: Reasoning-Oriented Prompting for a Planning Task

For planning and logic tasks, ask the model to provide a concise, checkable plan and final recommendation. Do not request hidden reasoning traces; focus on the visible steps and assumptions a user can evaluate.

In [ ]:
planning_prompt = """
Create a practical, checkable plan for a student who has 90 minutes to prepare
for a quiz on tokenization, embeddings, self-attention, and causal masking.

Return:
1. A time-boxed plan with activities totaling exactly 90 minutes.
2. One sentence explaining why the sequence of activities is effective.
3. Two assumptions you made.
Keep the response under 220 words.
"""

planning_results = show_comparison(planning_prompt)

# TODO: Compare at least two model responses when available.
# Evaluate whether each response totals 90 minutes and follows every constraint.

**Compare across models or prompt versions**

Use the questions below:

1. Which response followed all constraints most completely?
2. Which response was most useful for the intended student audience?
3. Did any response introduce an unsupported assumption?
4. What would you change in the prompt to correct the largest weakness?


## Section 7: Prompt Refinement Challenge

Start with the weak prompt below. Run it, identify the shortcomings, and revise it at least twice. Record the prompt and response after each iteration.

In [ ]:
weak_prompt = """
Make a study guide about transformers.
"""

print("Version 1")
version_1 = ask_model(weak_prompt, MODEL_NAMES[0])
print(version_1)

# TODO: Create Version 2 by adding a specific audience, scope, and length.
# prompt_v2 = """..."""
# version_2 = ask_model(prompt_v2, MODEL_NAMES[0])

# TODO: Create Version 3 by adding format requirements and quality constraints.
# prompt_v3 = """..."""
# version_3 = ask_model(prompt_v3, MODEL_NAMES[0])

In [ ]:
prompt_quality_rubric = pd.DataFrame(
    [
        ["Task clarity", "Does the prompt state a specific task?"],
        ["Audience and context", "Does it identify who needs the answer and why?"],
        ["Constraints", "Does it specify scope, length, tone, or exclusions?"],
        ["Output format", "Does it specify a usable format?"],
        ["Verifiability", "Can a reader check whether the answer met requirements?"],
    ],
    columns=["Criterion", "Question"],
)
prompt_quality_rubric

## Exit Ticket

Answer in a Markdown cell:

1. What is one advantage and one limitation of few-shot prompting?
2. Why should a structured output be validated after the model returns it?
3. Describe one revision that improved your prompt and explain why it helped.
4. When comparing models, what criteria matter besides whether the response sounds fluent?
5. Write one prompt-engineering principle you will use in future LLM work.

## Key Takeaway

A strong prompt is not just longer. It clearly communicates the task, relevant context, constraints, and desired output form, then is evaluated and refined against observable criteria.
